# 02 Build dataset

**Owner:** _TBD_

Build the one dataset every model uses: add OGT, replace each sequence by its ESM-2 embedding (480 numbers), and create the fixed train/test split.

| | |
| --- | --- |
| **Reads** | `data/interim/meltome.parquet`, `data/raw/ogt.csv`, `data/interim/embeddings.npy`, `data/interim/embeddings_index.parquet` |
| **Writes** | `data/processed/dataset.parquet`, `data/processed/split.csv` |

**Done when**

- [ ] `dataset.parquet` has the columns `id`, `organism`, `ogt`, `emb_0` … `emb_479`, `tm` in this order
- [ ] No missing values; 481 feature columns (`ogt` + 480 embedding dimensions)
- [ ] `split.csv` exists (`id`, `set` = `train` / `test`) and is committed

> **For AI agents:** follow `AGENTS.md` in the repository root. Do not recompute embeddings if the files exist. Never overwrite an existing `split.csv`.
> Only edit this notebook. Put a short markdown explanation above every code cell you fill in.

## 0. Setup

- Make the repository root importable, so `from src import ...` works whether Jupyter starts in the repo root or in `notebooks/` (add the root to `sys.path`).
- Import `src.paths` and use it for every file path. No hard-coded or absolute paths.
- Import the libraries this notebook needs.

## 1. Merge Meltome and OGT

- Load `meltome.parquet` and `ogt.csv`, join on `organism` (many proteins to one organism).
- Stop with a clear error if any protein has no OGT.

## 2. Cleaning decisions

Decide with the team and document under *Notes and decisions*:
- Keep both `lysate` and `cell` measurements? (All human rows are `cell`, all others `lysate`.)
- Sequences longer than 1022 residues are truncated for ESM-2. Keep them?
- Any Tm outliers to remove?

Apply the decisions here and report how many rows each step removes.

## 3. ESM-2 embeddings

- The embeddings already exist: copy `embeddings.npy`, `embeddings_index.parquet` and `embeddings.run.json` from the team drive into `data/interim/`.
- Only if they are missing, run `scripts/embed_esm.py` (ESM-2 35M, about 3.6 hours on CPU, resumes after an interruption).
- `embeddings_index.parquet` maps each `id` to a row in `embeddings.npy` (`embedding_row`) and stores `sequence_sha256`.
- Check that the SHA-256 of each sequence matches `sequence_sha256`; otherwise the embeddings belong to different sequences.

## 4. Assemble and save the dataset

- Build one table: `id`, `organism`, `ogt`, `emb_0` … `emb_479`, `tm`. The sequence itself is not included.
- Check the shape, missing values and the number of feature columns (481).
- Save to `paths.DATASET`.

## 5. Train/test split

- Only if `paths.SPLIT` does not exist: create the split with `src.data.make_split(dataset)` (random 80/20, stratified by organism, seed 42) and save it to `paths.SPLIT`.
- Show the number of train and test rows, overall and per organism.
- The split is shared by every model. Recreate it only if the whole team agrees, and commit `split.csv`.

## Notes and decisions

_Write down every choice made in this notebook (filters, parameters, problems, open questions) so teammates can follow it without reading the code._

-